# NMR Restraints Visualization Notebook

This notebook visualizes how deposited NMR restraints relate to a deposited structural model. It helps identify where restraint support is dense, sparse, or locally inconsistent.

Research-use scope only: this is an exploratory evidence viewer, not a formal validation engine. Restraint density is experimental coverage, not confidence, and violations are local inconsistencies, not automatic errors.

In [1]:
import base64
import hashlib
import io
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import gemmi
import numpy as np
import pandas as pd
import requests
from IPython.display import HTML, IFrame, display

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    widgets = None
    WIDGETS_AVAILABLE = False

try:
    import molviewspec as mvs
    MVS_AVAILABLE = True
except Exception as exc:
    mvs = None
    MVS_AVAILABLE = False
    MVS_IMPORT_ERROR = str(exc)

try:
    import pynmrstar
    PYNMRSTAR_AVAILABLE = True
except Exception as exc:
    pynmrstar = None
    PYNMRSTAR_AVAILABLE = False
    PYNMRSTAR_IMPORT_ERROR = str(exc)

RUN_START = time.time()
notebook_log: List[Dict[str, Any]] = []


def log_event(level: str, message: str, **meta: Any) -> None:
    notebook_log.append({
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "level": level,
        "message": message,
        **meta,
    })


def pkg_version(mod: Any) -> str:
    return getattr(mod, "__version__", "unknown") if mod is not None else "missing"


dependency_report = pd.DataFrame(
    [
        {"package": "python", "available": True, "version": f"{np.__class__.__module__.split('.')[0]} runtime"},
        {"package": "gemmi", "available": True, "version": pkg_version(gemmi)},
        {"package": "numpy", "available": True, "version": pkg_version(np)},
        {"package": "pandas", "available": True, "version": pkg_version(pd)},
        {"package": "requests", "available": True, "version": pkg_version(requests)},
        {"package": "pynmrstar", "available": PYNMRSTAR_AVAILABLE, "version": pkg_version(pynmrstar)},
        {"package": "molviewspec", "available": MVS_AVAILABLE, "version": pkg_version(mvs)},
        {"package": "ipywidgets", "available": WIDGETS_AVAILABLE, "version": pkg_version(widgets)},
    ]
)

display(dependency_report)
if not PYNMRSTAR_AVAILABLE:
    log_event("warning", "pynmrstar missing; restraint parsing will fail until installed", error=PYNMRSTAR_IMPORT_ERROR)
if not MVS_AVAILABLE:
    log_event("warning", "molviewspec missing; 3D rendering will use table fallbacks", error=globals().get("MVS_IMPORT_ERROR", ""))


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "specs").exists():
            return candidate
    return start.resolve()


PROJECT_ROOT = find_repo_root(Path.cwd())


,package,available,version
0,python,True,builtins runtime
1,gemmi,True,0.7.3
2,numpy,True,2.3.3
3,pandas,True,2.3.3
4,requests,True,2.32.5
5,pynmrstar,True,3.5.1
6,molviewspec,True,1.8.1
7,ipywidgets,True,8.1.7


In [2]:
CONFIG = {
    "input_mode": "remote",  # remote | local
    "pdb_id": "9L1V",
    "model_index": 0,
    "model_path": "",
    "restraint_path": "",
    "violation_threshold_distance": 0.5,
    "violation_threshold_dihedral": 5.0,
    "local_context_radius": 6.0,
    "max_visible_restraints": 250,
    "cache_enabled": True,
    "cache_root": str((PROJECT_ROOT / "specs/nmr_restraints/fixtures/cache").resolve()),
    "remote_model_endpoint": "https://www.ebi.ac.uk/pdbe/entry-files/download/{pdb_id}.cif",
    "remote_restraint_endpoint": "https://www.ebi.ac.uk/pdbe/entry-files/download/{pdb_id}_nmr-data.str",
    "fallback_model_endpoint": "https://files.rcsb.org/download/{pdb_id}.cif",
    "mapping_warn_threshold": 0.70,
    "mapping_normal_threshold": 0.95,
    "density_mode": "normalized",  # normalized | absolute
    "local_view_mode": "distance",  # distance | dihedral
}


def validate_pdb_id(value: str) -> str:
    if not isinstance(value, str) or len(value.strip()) != 4 or not value.strip().isalnum():
        raise ValueError("pdb_id must match ^[A-Za-z0-9]{4}$")
    return value.strip().upper()


def validate_config(config: Dict[str, Any]) -> Tuple[Dict[str, Any], Dict[str, Any], Dict[str, Any]]:
    cfg = dict(config)
    input_mode = str(cfg.get("input_mode", "remote")).strip().lower()
    if input_mode not in {"remote", "local"}:
        raise ValueError("input_mode must be one of: remote, local")

    if input_mode == "remote":
        cfg["pdb_id"] = validate_pdb_id(cfg.get("pdb_id", ""))
    else:
        model_path = Path(str(cfg.get("model_path", ""))).expanduser()
        restraint_path = Path(str(cfg.get("restraint_path", ""))).expanduser()
        if not model_path.exists() or model_path.suffix.lower() not in {".cif", ".mmcif"}:
            raise ValueError("local model_path must exist and end with .cif or .mmcif")
        if not restraint_path.exists() or restraint_path.suffix.lower() != ".str":
            raise ValueError("local restraint_path must exist and end with .str")
        cfg["model_path"] = str(model_path)
        cfg["restraint_path"] = str(restraint_path)

    cfg["model_index"] = int(cfg.get("model_index", 0))
    if cfg["model_index"] < 0:
        raise ValueError("model_index must be >= 0")

    cfg["violation_threshold_distance"] = float(cfg.get("violation_threshold_distance", 0.5))
    cfg["violation_threshold_dihedral"] = float(cfg.get("violation_threshold_dihedral", 5.0))
    cfg["local_context_radius"] = float(cfg.get("local_context_radius", 6.0))
    cfg["max_visible_restraints"] = int(cfg.get("max_visible_restraints", 250))

    if cfg["violation_threshold_distance"] < 0:
        raise ValueError("violation_threshold_distance must be >= 0")
    if not 0 <= cfg["violation_threshold_dihedral"] <= 180:
        raise ValueError("violation_threshold_dihedral must be between 0 and 180")
    if cfg["local_context_radius"] <= 0:
        raise ValueError("local_context_radius must be > 0")
    if cfg["max_visible_restraints"] < 0:
        raise ValueError("max_visible_restraints must be >= 0")

    input_request = {
        "input_mode": input_mode,
        "pdb_id": cfg.get("pdb_id"),
        "model_path": cfg.get("model_path", ""),
        "restraint_path": cfg.get("restraint_path", ""),
    }
    runtime_parameters = {
        "model_index": cfg["model_index"],
        "violation_threshold_distance": cfg["violation_threshold_distance"],
        "violation_threshold_dihedral": cfg["violation_threshold_dihedral"],
        "local_context_radius": cfg["local_context_radius"],
        "max_visible_restraints": cfg["max_visible_restraints"],
        "mapping_warn_threshold": float(cfg.get("mapping_warn_threshold", 0.70)),
        "mapping_normal_threshold": float(cfg.get("mapping_normal_threshold", 0.95)),
        "density_mode": str(cfg.get("density_mode", "normalized")),
        "local_view_mode": str(cfg.get("local_view_mode", "distance")),
    }

    stable_config = {k: v for k, v in cfg.items() if k not in {"model_path", "restraint_path"}}
    config_hash = hashlib.sha256(json.dumps(stable_config, sort_keys=True).encode("utf-8")).hexdigest()[:12]
    runtime_parameters["config_hash"] = config_hash
    return cfg, input_request, runtime_parameters


CONFIG, input_request, runtime_parameters = validate_config(CONFIG)

display(pd.DataFrame([{"parameter": k, "value": v} for k, v in CONFIG.items()]))


,parameter,value
0,input_mode,remote
1,pdb_id,9L1V
2,model_index,0
3,model_path,
4,restraint_path,
5,violation_threshold_distance,0.5
6,violation_threshold_dihedral,5.0
7,local_context_radius,6.0
8,max_visible_restraints,250
9,cache_enabled,True


## Input Modes, Defaults, And Scope

`remote` mode is the primary v1 path and defaults to `9L1V`. `local` mode is for parity/offline use with explicit file paths.

This notebook uses logical-restraint mapping coverage. For ambiguous distance groups (`Member_logic_code = OR`), a logical group counts as mapped when at least one candidate member maps and can be evaluated.

Distance rows missing either bound are diagnostics only in v1 and are excluded from geometry/violation evaluation.

In [3]:
def ensure_cache_dir(config: Dict[str, Any]) -> Path:
    root = Path(config.get("cache_root", "specs/nmr_restraints/fixtures/cache"))
    root.mkdir(parents=True, exist_ok=True)
    return root


def fetch_with_retries(url: str, timeout: int = 30, retries: int = 3) -> requests.Response:
    last_exc = None
    for attempt in range(retries):
        try:
            return requests.get(url, timeout=timeout)
        except Exception as exc:
            last_exc = exc
            wait = 2 ** attempt
            log_event("warning", "network fetch failed; retrying", url=url, attempt=attempt + 1, wait_seconds=wait, error=str(exc))
            time.sleep(wait)
    raise RuntimeError(f"failed to fetch after {retries} attempts: {url}; last error={last_exc}")


def sha256_path(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def resolve_model_source(config: Dict[str, Any], req: Dict[str, Any]) -> Dict[str, Any]:
    cache_root = ensure_cache_dir(config)
    now = datetime.now(timezone.utc).isoformat()

    if req["input_mode"] == "local":
        model_path = Path(req["model_path"])
        return {
            "status": "ok",
            "mode": "local-file",
            "path": str(model_path),
            "format": "mmcif",
            "source_url": None,
            "visualization_url": None,
            "timestamp_utc": now,
            "bytes": model_path.stat().st_size,
            "sha256": sha256_path(model_path),
            "from_cache": False,
        }

    pdb_id = req["pdb_id"].lower()
    cache_path = cache_root / f"{pdb_id}.cif"
    primary_url = config["remote_model_endpoint"].format(pdb_id=pdb_id)
    fallback_url = config["fallback_model_endpoint"].format(pdb_id=pdb_id)

    if bool(config.get("cache_enabled", True)) and cache_path.exists():
        return {
            "status": "ok",
            "mode": "cache",
            "path": str(cache_path),
            "format": "mmcif",
            "source_url": primary_url,
            "visualization_url": primary_url,
            "timestamp_utc": now,
            "bytes": cache_path.stat().st_size,
            "sha256": sha256_path(cache_path),
            "from_cache": True,
        }

    for url in (primary_url, fallback_url):
        try:
            response = fetch_with_retries(url)
            if response.status_code == 200 and response.content:
                cache_path.write_bytes(response.content)
                return {
                    "status": "ok",
                    "mode": "remote",
                    "path": str(cache_path),
                    "format": "mmcif",
                    "source_url": url,
                    "visualization_url": url,
                    "timestamp_utc": now,
                    "bytes": cache_path.stat().st_size,
                    "sha256": sha256_path(cache_path),
                    "from_cache": False,
                }
            log_event("warning", "model endpoint did not return 200", url=url, status_code=response.status_code)
        except Exception as exc:
            log_event("warning", "model retrieval error", url=url, error=str(exc))

    return {
        "status": "error",
        "mode": "remote",
        "path": None,
        "format": "mmcif",
        "source_url": primary_url,
        "visualization_url": primary_url,
        "timestamp_utc": now,
        "bytes": 0,
        "sha256": None,
        "from_cache": False,
        "error": "model retrieval failed from primary and fallback endpoints",
    }


def resolve_restraint_source(config: Dict[str, Any], req: Dict[str, Any]) -> Dict[str, Any]:
    cache_root = ensure_cache_dir(config)
    now = datetime.now(timezone.utc).isoformat()

    if req["input_mode"] == "local":
        restraint_path = Path(req["restraint_path"])
        return {
            "status": "ok",
            "mode": "local-file",
            "path": str(restraint_path),
            "source_url": None,
            "timestamp_utc": now,
            "bytes": restraint_path.stat().st_size,
            "sha256": sha256_path(restraint_path),
            "from_cache": False,
        }

    pdb_id = req["pdb_id"].lower()
    cache_path = cache_root / f"{pdb_id}_nmr-data.str"
    url = config["remote_restraint_endpoint"].format(pdb_id=pdb_id)

    if bool(config.get("cache_enabled", True)) and cache_path.exists():
        return {
            "status": "ok",
            "mode": "cache",
            "path": str(cache_path),
            "source_url": url,
            "timestamp_utc": now,
            "bytes": cache_path.stat().st_size,
            "sha256": sha256_path(cache_path),
            "from_cache": True,
        }

    try:
        response = fetch_with_retries(url)
        if response.status_code == 200 and response.content:
            cache_path.write_bytes(response.content)
            return {
                "status": "ok",
                "mode": "remote",
                "path": str(cache_path),
                "source_url": url,
                "timestamp_utc": now,
                "bytes": cache_path.stat().st_size,
                "sha256": sha256_path(cache_path),
                "from_cache": False,
            }
        return {
            "status": "missing",
            "mode": "remote",
            "path": None,
            "source_url": url,
            "timestamp_utc": now,
            "bytes": 0,
            "sha256": None,
            "from_cache": False,
            "http_status": response.status_code,
            "error": f"restraint endpoint returned {response.status_code}",
        }
    except Exception as exc:
        return {
            "status": "missing",
            "mode": "remote",
            "path": None,
            "source_url": url,
            "timestamp_utc": now,
            "bytes": 0,
            "sha256": None,
            "from_cache": False,
            "error": str(exc),
        }


model_source = resolve_model_source(CONFIG, input_request)
restraint_source = resolve_restraint_source(CONFIG, input_request)

provenance = pd.DataFrame([
    {
        "artifact": "model",
        "status": model_source.get("status"),
        "mode": model_source.get("mode"),
        "path": model_source.get("path"),
        "source_url": model_source.get("source_url"),
        "visualization_url": model_source.get("visualization_url"),
        "bytes": model_source.get("bytes"),
        "sha256": model_source.get("sha256"),
        "timestamp_utc": model_source.get("timestamp_utc"),
        "from_cache": model_source.get("from_cache"),
        "error": model_source.get("error", ""),
    },
    {
        "artifact": "restraints",
        "status": restraint_source.get("status"),
        "mode": restraint_source.get("mode"),
        "path": restraint_source.get("path"),
        "source_url": restraint_source.get("source_url"),
        "bytes": restraint_source.get("bytes"),
        "sha256": restraint_source.get("sha256"),
        "timestamp_utc": restraint_source.get("timestamp_utc"),
        "from_cache": restraint_source.get("from_cache"),
        "error": restraint_source.get("error", ""),
    },
])

if model_source.get("status") != "ok":
    raise RuntimeError(f"Model source unavailable: {model_source}")

display(provenance)


,artifact,status,mode,path,source_url,visualization_url,bytes,sha256,timestamp_utc,from_cache,error
0,model,ok,cache,/Users/mitsenkov/PycharmProjects/InsightFold/s...,https://www.ebi.ac.uk/pdbe/entry-files/downloa...,https://www.ebi.ac.uk/pdbe/entry-files/downloa...,2362171,e2ec15597e973bed4fa2673a53bf6756b9ee9bac805988...,2026-05-11T20:17:05.250730+00:00,True,
1,restraints,ok,cache,/Users/mitsenkov/PycharmProjects/InsightFold/s...,https://www.ebi.ac.uk/pdbe/entry-files/downloa...,NaN,413575,b1c1535358df907f732e9946b8148ab9d4f0a130cd445f...,2026-05-11T20:17:05.253448+00:00,True,


In [4]:
def normalize_seqid(value: Any) -> str:
    if value is None:
        return ""
    text = str(value).strip()
    return text


def parse_structure_with_gemmi(model_source: Dict[str, Any], model_index: int) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[Tuple[str, str, str, str], Dict[str, Any]], Dict[str, Any]]:
    structure = gemmi.read_structure(model_source["path"])
    if model_index < 0 or model_index >= len(structure):
        raise IndexError(f"model_index={model_index} outside available range 0..{len(structure)-1}")

    model = structure[model_index]
    atom_rows: List[Dict[str, Any]] = []
    residue_rows: List[Dict[str, Any]] = []
    atom_lookup: Dict[Tuple[str, str, str, str], Dict[str, Any]] = {}
    ignored_altloc = 0

    for cra in model.all():
        chain = cra.chain
        residue = cra.residue
        atom = cra.atom

        alt = str(atom.altloc).replace("\x00", "").strip()
        include = alt in {"", ".", "?", "A"}
        if not include:
            ignored_altloc += 1
            continue

        auth_asym_id = chain.name
        auth_seq_id = normalize_seqid(residue.seqid)
        auth_comp_id = residue.name
        auth_atom_id = atom.name.strip()

        row = {
            "model_index": model_index,
            "chain_id": chain.name,
            "auth_asym_id": auth_asym_id,
            "auth_seq_id": auth_seq_id,
            "auth_comp_id": auth_comp_id,
            "auth_atom_id": auth_atom_id,
            "label_asym_id": chain.name,
            "label_seq_id": residue.label_seq,
            "x": float(atom.pos.x),
            "y": float(atom.pos.y),
            "z": float(atom.pos.z),
            "element": atom.element.name,
            "altloc": alt,
            "included_altloc": include,
        }
        atom_rows.append(row)

        key = (auth_asym_id, auth_seq_id, auth_comp_id, auth_atom_id)
        atom_lookup[key] = row

    atom_table = pd.DataFrame(atom_rows)

    if atom_table.empty:
        raise RuntimeError("atom_table is empty after parsing model")

    for (auth_asym_id, auth_seq_id, auth_comp_id), group in atom_table.groupby(["auth_asym_id", "auth_seq_id", "auth_comp_id"], sort=False):
        residue_key = f"{auth_asym_id}:{auth_seq_id}:{auth_comp_id}:{model_index}"

        ca = group[group["auth_atom_id"] == "CA"]
        if not ca.empty:
            ca_row = ca.iloc[0]
            ca_xyz = (float(ca_row["x"]), float(ca_row["y"]), float(ca_row["z"]))
        else:
            ca_xyz = (np.nan, np.nan, np.nan)

        centroid = (float(group["x"].mean()), float(group["y"].mean()), float(group["z"].mean()))

        residue_rows.append({
            "residue_key": residue_key,
            "chain_id": auth_asym_id,
            "auth_asym_id": auth_asym_id,
            "auth_seq_id": auth_seq_id,
            "auth_comp_id": auth_comp_id,
            "label_asym_id": group.iloc[0]["label_asym_id"],
            "label_seq_id": group.iloc[0]["label_seq_id"],
            "ca_x": ca_xyz[0],
            "ca_y": ca_xyz[1],
            "ca_z": ca_xyz[2],
            "centroid_x": centroid[0],
            "centroid_y": centroid[1],
            "centroid_z": centroid[2],
            "atom_count": int(len(group)),
            "residue_type": "protein",
        })

    residue_table = pd.DataFrame(residue_rows)

    block = structure.make_mmcif_document().sole_block()
    exptl_method = ""
    submitted_models = None
    try:
        vals = block.find_values("_exptl.method")
        exptl_method = vals[0] if vals else ""
    except Exception:
        exptl_method = ""
    try:
        vals = block.find_values("_pdbx_nmr_ensemble.conformers_submitted_total_number")
        submitted_models = int(vals[0]) if vals and str(vals[0]).strip() not in {".", "?", ""} else None
    except Exception:
        submitted_models = None

    model_summary = {
        "model_index": model_index,
        "model_count": len(structure),
        "submitted_models": submitted_models,
        "exptl_method": exptl_method,
        "atom_count": int(len(atom_table)),
        "residue_count": int(len(residue_table)),
        "chains": sorted(residue_table["auth_asym_id"].dropna().unique().tolist()),
        "ignored_altloc_count": ignored_altloc,
    }
    return atom_table, residue_table, atom_lookup, model_summary


atom_table, residue_table, atom_lookup, model_summary = parse_structure_with_gemmi(model_source, runtime_parameters["model_index"])
display(pd.DataFrame([model_summary]))
display(atom_table.head(3))
display(residue_table.head(3))


,model_index,model_count,submitted_models,exptl_method,atom_count,residue_count,chains,ignored_altloc_count
0,0,20,None,'SOLUTION NMR',1330,82,[A],0


,model_index,chain_id,auth_asym_id,auth_seq_id,auth_comp_id,auth_atom_id,label_asym_id,label_seq_id,x,y,z,element,altloc,included_altloc
0,0,A,A,1,MET,N,A,1,-12.866,8.925,8.636,N,,True
1,0,A,A,1,MET,CA,A,1,-12.582,7.510,8.262,C,,True
2,0,A,A,1,MET,C,A,1,-12.441,7.367,6.769,C,,True


,residue_key,chain_id,auth_asym_id,auth_seq_id,auth_comp_id,label_asym_id,label_seq_id,ca_x,ca_y,ca_z,centroid_x,centroid_y,centroid_z,atom_count,residue_type
0,A:1:MET:0,A,A,1,MET,A,1,-12.582,7.510,8.262,-13.705316,6.833211,9.371474,19,protein
1,A:2:LEU:0,A,A,2,LEU,A,2,-11.236,6.374,4.914,-10.009263,7.547211,4.653158,19,protein
2,A:3:SER:0,A,A,3,SER,A,3,-12.059,3.223,2.991,-12.583091,3.381636,2.412818,11,protein


In [5]:
DISTANCE_TAGS_REQUIRED = [
    "_Gen_dist_constraint.ID",
    "_Gen_dist_constraint.Member_ID",
    "_Gen_dist_constraint.Member_logic_code",
    "_Gen_dist_constraint.Auth_asym_ID_1",
    "_Gen_dist_constraint.Auth_seq_ID_1",
    "_Gen_dist_constraint.Auth_comp_ID_1",
    "_Gen_dist_constraint.Auth_atom_ID_1",
    "_Gen_dist_constraint.Auth_asym_ID_2",
    "_Gen_dist_constraint.Auth_seq_ID_2",
    "_Gen_dist_constraint.Auth_comp_ID_2",
    "_Gen_dist_constraint.Auth_atom_ID_2",
    "_Gen_dist_constraint.Distance_lower_bound_val",
    "_Gen_dist_constraint.Distance_upper_bound_val",
]

DIHEDRAL_TAGS_REQUIRED = [
    "_Torsion_angle_constraint.ID",
    "_Torsion_angle_constraint.Torsion_angle_name",
    "_Torsion_angle_constraint.Auth_asym_ID_1",
    "_Torsion_angle_constraint.Auth_seq_ID_1",
    "_Torsion_angle_constraint.Auth_comp_ID_1",
    "_Torsion_angle_constraint.Auth_atom_ID_1",
    "_Torsion_angle_constraint.Auth_asym_ID_2",
    "_Torsion_angle_constraint.Auth_seq_ID_2",
    "_Torsion_angle_constraint.Auth_comp_ID_2",
    "_Torsion_angle_constraint.Auth_atom_ID_2",
    "_Torsion_angle_constraint.Auth_asym_ID_3",
    "_Torsion_angle_constraint.Auth_seq_ID_3",
    "_Torsion_angle_constraint.Auth_comp_ID_3",
    "_Torsion_angle_constraint.Auth_atom_ID_3",
    "_Torsion_angle_constraint.Auth_asym_ID_4",
    "_Torsion_angle_constraint.Auth_seq_ID_4",
    "_Torsion_angle_constraint.Auth_comp_ID_4",
    "_Torsion_angle_constraint.Auth_atom_ID_4",
    "_Torsion_angle_constraint.Angle_lower_bound_val",
    "_Torsion_angle_constraint.Angle_upper_bound_val",
]


def parse_nmrstar_restraints(restraint_source: Dict[str, Any]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    if restraint_source.get("status") != "ok":
        unsupported = pd.DataFrame([
            {
                "category": "missing_restraints",
                "reason": restraint_source.get("error", "restraint source unavailable"),
                "source_saveframe": None,
                "source_loop": None,
                "row_count": 0,
            }
        ])
        diagnostics = {
            "distance_loops": 0,
            "distance_rows": 0,
            "distance_logical": 0,
            "distance_or_logical": 0,
            "torsion_loops": 0,
            "torsion_rows": 0,
            "torsion_logical": 0,
            "distance_missing_bounds": 0,
            "torsion_missing_bounds": 0,
            "wrapped_torsion_bounds": 0,
            "parser": "pynmrstar_missing" if not PYNMRSTAR_AVAILABLE else "pynmrstar",
        }
        return pd.DataFrame(), pd.DataFrame(), unsupported, diagnostics

    if not PYNMRSTAR_AVAILABLE:
        raise RuntimeError("pynmrstar is required for restraint parsing in this notebook")

    entry = pynmrstar.Entry.from_file(restraint_source["path"])
    distance_frames: List[pd.DataFrame] = []
    torsion_frames: List[pd.DataFrame] = []
    unsupported_records: List[Dict[str, Any]] = []

    distance_loop_count = 0
    torsion_loop_count = 0

    for saveframe in entry.frame_list:
        for loop in saveframe.loops:
            tags = list(loop.get_tag_names())
            tag_set = set(tags)

            is_distance = any(t.startswith("_Gen_dist_constraint.") for t in tags)
            is_torsion = any(t.startswith("_Torsion_angle_constraint.") for t in tags)

            if is_distance:
                missing = [t for t in DISTANCE_TAGS_REQUIRED if t not in tag_set]
                if missing:
                    unsupported_records.append({
                        "category": "distance_loop_unsupported",
                        "reason": "missing_required_tags",
                        "missing_tags": ",".join(missing),
                        "source_saveframe": saveframe.name,
                        "source_loop": "_Gen_dist_constraint",
                        "row_count": len(loop.data),
                    })
                    continue

                frame = pd.DataFrame(loop.data, columns=tags)
                frame["source_saveframe"] = saveframe.name
                frame["source_loop"] = "_Gen_dist_constraint"
                distance_frames.append(frame)
                distance_loop_count += 1

            if is_torsion:
                missing = [t for t in DIHEDRAL_TAGS_REQUIRED if t not in tag_set]
                if missing:
                    unsupported_records.append({
                        "category": "torsion_loop_unsupported",
                        "reason": "missing_required_tags",
                        "missing_tags": ",".join(missing),
                        "source_saveframe": saveframe.name,
                        "source_loop": "_Torsion_angle_constraint",
                        "row_count": len(loop.data),
                    })
                    continue

                frame = pd.DataFrame(loop.data, columns=tags)
                frame["source_saveframe"] = saveframe.name
                frame["source_loop"] = "_Torsion_angle_constraint"
                torsion_frames.append(frame)
                torsion_loop_count += 1

    if distance_frames:
        distance = pd.concat(distance_frames, ignore_index=True)
    else:
        distance = pd.DataFrame()

    if torsion_frames:
        torsion = pd.concat(torsion_frames, ignore_index=True)
    else:
        torsion = pd.DataFrame()

    if not distance.empty:
        dist_cols = {
            "_Gen_dist_constraint.ID": "restraint_id",
            "_Gen_dist_constraint.Member_ID": "member_id",
            "_Gen_dist_constraint.Member_logic_code": "member_logic_code",
            "_Gen_dist_constraint.Auth_asym_ID_1": "auth_asym_id_1",
            "_Gen_dist_constraint.Auth_seq_ID_1": "auth_seq_id_1",
            "_Gen_dist_constraint.Auth_comp_ID_1": "auth_comp_id_1",
            "_Gen_dist_constraint.Auth_atom_ID_1": "auth_atom_id_1",
            "_Gen_dist_constraint.Auth_asym_ID_2": "auth_asym_id_2",
            "_Gen_dist_constraint.Auth_seq_ID_2": "auth_seq_id_2",
            "_Gen_dist_constraint.Auth_comp_ID_2": "auth_comp_id_2",
            "_Gen_dist_constraint.Auth_atom_ID_2": "auth_atom_id_2",
            "_Gen_dist_constraint.Distance_lower_bound_val": "lower_bound",
            "_Gen_dist_constraint.Distance_upper_bound_val": "upper_bound",
            "_Gen_dist_constraint.Gen_dist_constraint_list_ID": "source_list_id",
        }
        distance = distance.rename(columns=dist_cols)
        for col in [
            "restraint_id", "member_id", "member_logic_code",
            "auth_asym_id_1", "auth_seq_id_1", "auth_comp_id_1", "auth_atom_id_1",
            "auth_asym_id_2", "auth_seq_id_2", "auth_comp_id_2", "auth_atom_id_2",
            "source_saveframe", "source_loop", "source_list_id",
        ]:
            if col in distance.columns:
                distance[col] = distance[col].astype(str).str.strip()
        distance["lower_bound"] = pd.to_numeric(distance["lower_bound"], errors="coerce")
        distance["upper_bound"] = pd.to_numeric(distance["upper_bound"], errors="coerce")

    if not torsion.empty:
        tor_cols = {
            "_Torsion_angle_constraint.ID": "restraint_id",
            "_Torsion_angle_constraint.Torsion_angle_name": "torsion_name",
            "_Torsion_angle_constraint.Auth_asym_ID_1": "auth_asym_id_1",
            "_Torsion_angle_constraint.Auth_seq_ID_1": "auth_seq_id_1",
            "_Torsion_angle_constraint.Auth_comp_ID_1": "auth_comp_id_1",
            "_Torsion_angle_constraint.Auth_atom_ID_1": "auth_atom_id_1",
            "_Torsion_angle_constraint.Auth_asym_ID_2": "auth_asym_id_2",
            "_Torsion_angle_constraint.Auth_seq_ID_2": "auth_seq_id_2",
            "_Torsion_angle_constraint.Auth_comp_ID_2": "auth_comp_id_2",
            "_Torsion_angle_constraint.Auth_atom_ID_2": "auth_atom_id_2",
            "_Torsion_angle_constraint.Auth_asym_ID_3": "auth_asym_id_3",
            "_Torsion_angle_constraint.Auth_seq_ID_3": "auth_seq_id_3",
            "_Torsion_angle_constraint.Auth_comp_ID_3": "auth_comp_id_3",
            "_Torsion_angle_constraint.Auth_atom_ID_3": "auth_atom_id_3",
            "_Torsion_angle_constraint.Auth_asym_ID_4": "auth_asym_id_4",
            "_Torsion_angle_constraint.Auth_seq_ID_4": "auth_seq_id_4",
            "_Torsion_angle_constraint.Auth_comp_ID_4": "auth_comp_id_4",
            "_Torsion_angle_constraint.Auth_atom_ID_4": "auth_atom_id_4",
            "_Torsion_angle_constraint.Angle_lower_bound_val": "lower_bound",
            "_Torsion_angle_constraint.Angle_upper_bound_val": "upper_bound",
            "_Torsion_angle_constraint.Angle_target_val": "target_angle",
            "_Torsion_angle_constraint.Torsion_angle_constraint_list_ID": "source_list_id",
        }
        torsion = torsion.rename(columns=tor_cols)
        for col in [
            "restraint_id", "torsion_name",
            "auth_asym_id_1", "auth_seq_id_1", "auth_comp_id_1", "auth_atom_id_1",
            "auth_asym_id_2", "auth_seq_id_2", "auth_comp_id_2", "auth_atom_id_2",
            "auth_asym_id_3", "auth_seq_id_3", "auth_comp_id_3", "auth_atom_id_3",
            "auth_asym_id_4", "auth_seq_id_4", "auth_comp_id_4", "auth_atom_id_4",
            "source_saveframe", "source_loop", "source_list_id",
        ]:
            if col in torsion.columns:
                torsion[col] = torsion[col].astype(str).str.strip()
        torsion["lower_bound"] = pd.to_numeric(torsion["lower_bound"], errors="coerce")
        torsion["upper_bound"] = pd.to_numeric(torsion["upper_bound"], errors="coerce")
        torsion["target_angle"] = pd.to_numeric(torsion.get("target_angle"), errors="coerce")

    distance_missing = int(distance[["lower_bound", "upper_bound"]].isna().any(axis=1).sum()) if not distance.empty else 0
    torsion_missing = int(torsion[["lower_bound", "upper_bound"]].isna().any(axis=1).sum()) if not torsion.empty else 0

    if distance_missing:
        unsupported_records.append({
            "category": "distance_missing_bounds",
            "reason": "distance rows missing lower or upper bound",
            "source_saveframe": None,
            "source_loop": "_Gen_dist_constraint",
            "row_count": distance_missing,
        })

    if torsion_missing:
        unsupported_records.append({
            "category": "torsion_missing_bounds",
            "reason": "torsion rows missing lower or upper bound",
            "source_saveframe": None,
            "source_loop": "_Torsion_angle_constraint",
            "row_count": torsion_missing,
        })

    wrapped_count = 0
    if not torsion.empty:
        wrapped_count = int((torsion["lower_bound"] > torsion["upper_bound"]).sum())

    unsupported = pd.DataFrame(unsupported_records)
    diagnostics = {
        "distance_loops": distance_loop_count,
        "distance_rows": int(len(distance)),
        "distance_logical": int(distance["restraint_id"].nunique()) if not distance.empty else 0,
        "distance_or_logical": int(distance.loc[distance["member_logic_code"].str.upper() == "OR", "restraint_id"].nunique()) if not distance.empty else 0,
        "torsion_loops": torsion_loop_count,
        "torsion_rows": int(len(torsion)),
        "torsion_logical": int(torsion["restraint_id"].nunique()) if not torsion.empty else 0,
        "distance_missing_bounds": distance_missing,
        "torsion_missing_bounds": torsion_missing,
        "wrapped_torsion_bounds": wrapped_count,
        "parser": f"pynmrstar {pynmrstar.__version__}" if PYNMRSTAR_AVAILABLE else "missing",
    }
    return distance, torsion, unsupported, diagnostics


distance_restraints_raw, dihedral_restraints_raw, unsupported_records, restraint_diagnostics = parse_nmrstar_restraints(restraint_source)

print("Restraint diagnostics:")
display(pd.DataFrame([restraint_diagnostics]))
if not distance_restraints_raw.empty:
    display(distance_restraints_raw.head(3))
if not dihedral_restraints_raw.empty:
    display(dihedral_restraints_raw.head(3))
if not unsupported_records.empty:
    display(unsupported_records)


Restraint diagnostics:


,distance_loops,distance_rows,distance_logical,distance_or_logical,torsion_loops,torsion_rows,torsion_logical,distance_missing_bounds,torsion_missing_bounds,wrapped_torsion_bounds,parser
0,2,2219,1150,451,1,149,149,0,0,0,pynmrstar 3.5.1


,_Gen_dist_constraint.Index_ID,restraint_id,_Gen_dist_constraint.Combination_ID,member_id,member_logic_code,_Gen_dist_constraint.Entity_assembly_ID_1,_Gen_dist_constraint.Entity_ID_1,_Gen_dist_constraint.Comp_index_ID_1,_Gen_dist_constraint.Comp_ID_1,_Gen_dist_constraint.Atom_ID_1,...,_Gen_dist_constraint.Auth_atom_name_1,auth_asym_id_2,auth_seq_id_2,auth_comp_id_2,auth_atom_id_2,_Gen_dist_constraint.Auth_atom_name_2,_Gen_dist_constraint.Entry_ID,source_list_id,source_saveframe,source_loop
0,1,1,.,.,OR,1,1,1,MET,HA,...,HA*,A,80,THR,HG21,HG2*,9L1V,1,XPLOR-NIH_distance_restraints_1,_Gen_dist_constraint
1,2,1,.,.,OR,1,1,1,MET,HA,...,HA*,A,80,THR,HG22,HG2*,9L1V,1,XPLOR-NIH_distance_restraints_1,_Gen_dist_constraint
2,3,1,.,.,OR,1,1,1,MET,HA,...,HA*,A,80,THR,HG23,HG2*,9L1V,1,XPLOR-NIH_distance_restraints_1,_Gen_dist_constraint


,_Torsion_angle_constraint.Index_ID,restraint_id,_Torsion_angle_constraint.Combination_ID,torsion_name,_Torsion_angle_constraint.Entity_assembly_ID_1,_Torsion_angle_constraint.Entity_ID_1,_Torsion_angle_constraint.Comp_index_ID_1,_Torsion_angle_constraint.Comp_ID_1,_Torsion_angle_constraint.Atom_ID_1,_Torsion_angle_constraint.Entity_assembly_ID_2,...,_Torsion_angle_constraint.Auth_atom_name_3,auth_asym_id_4,auth_seq_id_4,auth_comp_id_4,auth_atom_id_4,_Torsion_angle_constraint.Auth_atom_name_4,_Torsion_angle_constraint.Entry_ID,source_list_id,source_saveframe,source_loop
0,1,1,.,PHI,1,1,1,MET,C,1,...,CA,A,2,LEU,C,C,9L1V,1,XPLOR-NIH_dihedral_angle_restraints_1,_Torsion_angle_constraint
1,2,2,.,PSI,1,1,2,LEU,N,1,...,C,A,3,SER,N,N,9L1V,1,XPLOR-NIH_dihedral_angle_restraints_1,_Torsion_angle_constraint
2,3,3,.,PHI,1,1,2,LEU,C,1,...,CA,A,3,SER,C,C,9L1V,1,XPLOR-NIH_dihedral_angle_restraints_1,_Torsion_angle_constraint


In [6]:
def atom_key_from_row(row: pd.Series, suffix: str) -> Tuple[str, str, str, str]:
    return (
        str(row[f"auth_asym_id_{suffix}"]).strip(),
        normalize_seqid(row[f"auth_seq_id_{suffix}"]),
        str(row[f"auth_comp_id_{suffix}"]).strip(),
        str(row[f"auth_atom_id_{suffix}"]).strip(),
    )


def residue_key_from_atom_key(atom_key: Tuple[str, str, str, str], model_index: int) -> str:
    return f"{atom_key[0]}:{atom_key[1]}:{atom_key[2]}:{model_index}"


def map_distance_restraints(distance: pd.DataFrame, atom_lookup: Dict[Tuple[str, str, str, str], Dict[str, Any]], model_index: int) -> pd.DataFrame:
    if distance.empty:
        return distance.copy()

    mapped = distance.copy()
    key1 = mapped.apply(lambda row: atom_key_from_row(row, "1"), axis=1)
    key2 = mapped.apply(lambda row: atom_key_from_row(row, "2"), axis=1)

    mapped["atom_key_1"] = key1
    mapped["atom_key_2"] = key2
    mapped["is_supported_bounds"] = ~(mapped[["lower_bound", "upper_bound"]].isna().any(axis=1))
    mapped["coord_1"] = key1.map(lambda k: (atom_lookup[k]["x"], atom_lookup[k]["y"], atom_lookup[k]["z"]) if k in atom_lookup else None)
    mapped["coord_2"] = key2.map(lambda k: (atom_lookup[k]["x"], atom_lookup[k]["y"], atom_lookup[k]["z"]) if k in atom_lookup else None)
    mapped["mapped_member"] = mapped["coord_1"].notna() & mapped["coord_2"].notna() & mapped["is_supported_bounds"]

    mapped["unmapped_reason"] = ""
    mapped.loc[mapped["coord_1"].isna(), "unmapped_reason"] = "atom_1_not_found"
    mapped.loc[mapped["coord_2"].isna(), "unmapped_reason"] = mapped.loc[mapped["coord_2"].isna(), "unmapped_reason"].replace("", "atom_2_not_found")
    mapped.loc[~mapped["is_supported_bounds"], "unmapped_reason"] = mapped.loc[~mapped["is_supported_bounds"], "unmapped_reason"].replace("", "missing_bounds")

    mapped["mapping_status"] = np.where(mapped["mapped_member"], "mapped", "unmapped")
    mapped["residue_key_1"] = mapped["atom_key_1"].map(lambda k: residue_key_from_atom_key(k, model_index))
    mapped["residue_key_2"] = mapped["atom_key_2"].map(lambda k: residue_key_from_atom_key(k, model_index))
    return mapped


def map_dihedral_restraints(dihedral: pd.DataFrame, atom_lookup: Dict[Tuple[str, str, str, str], Dict[str, Any]], model_index: int) -> pd.DataFrame:
    if dihedral.empty:
        return dihedral.copy()

    mapped = dihedral.copy()
    for idx in ["1", "2", "3", "4"]:
        mapped[f"atom_key_{idx}"] = mapped.apply(lambda row: atom_key_from_row(row, idx), axis=1)
        mapped[f"coord_{idx}"] = mapped[f"atom_key_{idx}"].map(
            lambda k: (atom_lookup[k]["x"], atom_lookup[k]["y"], atom_lookup[k]["z"]) if k in atom_lookup else None
        )
        mapped[f"residue_key_{idx}"] = mapped[f"atom_key_{idx}"].map(lambda k: residue_key_from_atom_key(k, model_index))

    mapped["is_supported_bounds"] = ~(mapped[["lower_bound", "upper_bound"]].isna().any(axis=1))
    mapped["mapped_member"] = mapped[["coord_1", "coord_2", "coord_3", "coord_4"]].notna().all(axis=1) & mapped["is_supported_bounds"]
    mapped["mapping_status"] = np.where(mapped["mapped_member"], "mapped", "unmapped")
    mapped["unmapped_reason"] = ""
    for idx in ["1", "2", "3", "4"]:
        col = f"coord_{idx}"
        mapped.loc[mapped[col].isna(), "unmapped_reason"] = mapped.loc[mapped[col].isna(), "unmapped_reason"].replace("", f"atom_{idx}_not_found")
    mapped.loc[~mapped["is_supported_bounds"], "unmapped_reason"] = mapped.loc[~mapped["is_supported_bounds"], "unmapped_reason"].replace("", "missing_bounds")
    return mapped


def compute_mapping_report(mapped_distance: pd.DataFrame, mapped_dihedral: pd.DataFrame, config: Dict[str, Any]) -> Dict[str, Any]:
    logical_records = []

    if not mapped_distance.empty:
        for rid, group in mapped_distance.groupby("restraint_id", sort=False):
            logic_codes = group["member_logic_code"].fillna("").str.upper()
            is_or = bool((logic_codes == "OR").any())
            logical_mapped = bool(group["mapped_member"].any()) if is_or else bool(group["mapped_member"].any())
            logical_records.append({
                "logical_id": f"distance:{rid}",
                "restraint_type": "distance",
                "is_or": is_or,
                "logical_mapped": logical_mapped,
            })

    if not mapped_dihedral.empty:
        for rid, group in mapped_dihedral.groupby("restraint_id", sort=False):
            logical_mapped = bool(group["mapped_member"].any())
            logical_records.append({
                "logical_id": f"dihedral:{rid}",
                "restraint_type": "dihedral",
                "is_or": False,
                "logical_mapped": logical_mapped,
            })

    logical_df = pd.DataFrame(logical_records)
    parsed_logical = int(len(logical_df))
    mapped_logical = int(logical_df["logical_mapped"].sum()) if not logical_df.empty else 0
    unmapped_logical = parsed_logical - mapped_logical
    coverage = (mapped_logical / parsed_logical) if parsed_logical else 0.0

    warn_threshold = float(config.get("mapping_warn_threshold", 0.70))
    normal_threshold = float(config.get("mapping_normal_threshold", 0.95))
    if coverage < warn_threshold:
        status = "abort_low_mapping"
    elif coverage < normal_threshold:
        status = "warning_partial_mapping"
    else:
        status = "normal"

    report = {
        "parsed_logical_restraints": parsed_logical,
        "mapped_logical_restraints": mapped_logical,
        "unmapped_logical_restraints": unmapped_logical,
        "logical_mapping_coverage": coverage,
        "threshold_status": status,
        "distance_member_rows": int(len(mapped_distance)) if not mapped_distance.empty else 0,
        "distance_member_mapped": int(mapped_distance["mapped_member"].sum()) if not mapped_distance.empty else 0,
        "distance_member_unmapped": int((~mapped_distance["mapped_member"]).sum()) if not mapped_distance.empty else 0,
        "distance_or_logical_restraints": int(mapped_distance.loc[mapped_distance["member_logic_code"].str.upper() == "OR", "restraint_id"].nunique()) if not mapped_distance.empty else 0,
        "dihedral_member_rows": int(len(mapped_dihedral)) if not mapped_dihedral.empty else 0,
        "dihedral_member_mapped": int(mapped_dihedral["mapped_member"].sum()) if not mapped_dihedral.empty else 0,
        "dihedral_member_unmapped": int((~mapped_dihedral["mapped_member"]).sum()) if not mapped_dihedral.empty else 0,
    }

    if not mapped_distance.empty:
        sample = mapped_distance.loc[~mapped_distance["mapped_member"], ["restraint_id", "member_id", "unmapped_reason"]].head(5)
        report["unmapped_distance_examples"] = sample.to_dict(orient="records")
    else:
        report["unmapped_distance_examples"] = []

    return report


mapped_distance_restraints = map_distance_restraints(distance_restraints_raw, atom_lookup, runtime_parameters["model_index"])
mapped_dihedral_restraints = map_dihedral_restraints(dihedral_restraints_raw, atom_lookup, runtime_parameters["model_index"])
mapping_report = compute_mapping_report(mapped_distance_restraints, mapped_dihedral_restraints, runtime_parameters)

display(pd.DataFrame([mapping_report]))
if mapping_report["unmapped_distance_examples"]:
    display(pd.DataFrame(mapping_report["unmapped_distance_examples"]))


,parsed_logical_restraints,mapped_logical_restraints,unmapped_logical_restraints,logical_mapping_coverage,threshold_status,distance_member_rows,distance_member_mapped,distance_member_unmapped,distance_or_logical_restraints,dihedral_member_rows,dihedral_member_mapped,dihedral_member_unmapped,unmapped_distance_examples
0,1299,1299,0,1.0,normal,2219,2219,0,451,149,149,0,[]


In [7]:
def distance_violation(measured: float, lower: float, upper: float) -> float:
    if measured < lower:
        return float(lower - measured)
    if measured > upper:
        return float(measured - upper)
    return 0.0


def normalize_angle(angle: float) -> float:
    val = ((float(angle) + 180.0) % 360.0) - 180.0
    if val == -180.0:
        return 180.0
    return val


def circular_distance_deg(a: float, b: float) -> float:
    diff = abs(normalize_angle(a - b))
    return min(diff, 360.0 - diff)


def dihedral_violation(angle: float, lower: float, upper: float) -> float:
    angle = normalize_angle(angle)
    lower = normalize_angle(lower)
    upper = normalize_angle(upper)

    if lower <= upper:
        if lower <= angle <= upper:
            return 0.0
        return min(circular_distance_deg(angle, lower), circular_distance_deg(angle, upper))

    # wrapped interval, e.g. 170 to -170
    in_interval = (angle >= lower) or (angle <= upper)
    if in_interval:
        return 0.0
    return min(circular_distance_deg(angle, lower), circular_distance_deg(angle, upper))


def compute_dihedral(coords: np.ndarray) -> float:
    p0, p1, p2, p3 = coords
    b0 = p0 - p1
    b1 = p2 - p1
    b2 = p3 - p2
    b1_norm = b1 / np.linalg.norm(b1)
    v = b0 - np.dot(b0, b1_norm) * b1_norm
    w = b2 - np.dot(b2, b1_norm) * b1_norm
    x = np.dot(v, w)
    y = np.dot(np.cross(b1_norm, v), w)
    return normalize_angle(np.degrees(np.arctan2(y, x)))


def color_class(magnitude: float, threshold: float) -> str:
    if magnitude <= 0:
        return "none"
    if magnitude < threshold:
        return "low"
    if magnitude < 2 * threshold:
        return "medium"
    return "high"


def evaluate_restraints(
    mapped_distance: pd.DataFrame,
    mapped_dihedral: pd.DataFrame,
    residue_table: pd.DataFrame,
    mapping_report: Dict[str, Any],
    params: Dict[str, Any],
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Any], Dict[str, Any]]:
    if mapping_report["threshold_status"] == "abort_low_mapping":
        geometry_diagnostics = {
            "analysis_status": "aborted_low_mapping",
            "distance_evaluated": 0,
            "dihedral_evaluated": 0,
        }
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), geometry_diagnostics, {
            "residue_count": int(len(residue_table)),
            "logical_restraints_count": 0,
            "structure_mean_density": 0.0,
            "max_absolute_density": 0,
        }

    evaluated_rows: List[Dict[str, Any]] = []
    distance_threshold = float(params["violation_threshold_distance"])
    dihedral_threshold = float(params["violation_threshold_dihedral"])

    if not mapped_distance.empty:
        for restraint_id, group in mapped_distance.groupby("restraint_id", sort=False):
            logic_codes = group["member_logic_code"].fillna("").str.upper()
            is_or = bool((logic_codes == "OR").any())

            mapped_group = group[group["mapped_member"]].copy()
            if mapped_group.empty:
                continue

            measured = []
            for idx, row in mapped_group.iterrows():
                c1 = np.array(row["coord_1"], dtype=float)
                c2 = np.array(row["coord_2"], dtype=float)
                d = float(np.linalg.norm(c1 - c2))
                measured.append((idx, d))

            if is_or:
                selected_idx, selected_distance = min(measured, key=lambda x: x[1])
                selected = mapped_group.loc[selected_idx]
            else:
                selected_idx, selected_distance = measured[0]
                selected = mapped_group.loc[selected_idx]

            lb = float(selected["lower_bound"])
            ub = float(selected["upper_bound"])
            violation = distance_violation(selected_distance, lb, ub)
            involved = sorted({selected["residue_key_1"], selected["residue_key_2"]})
            selected_member = selected.get("member_id", "")

            evaluated_rows.append({
                "restraint_id": str(restraint_id),
                "restraint_type": "distance",
                "logical_restraint_key": f"distance:{restraint_id}",
                "ambiguity_type": "OR" if is_or else "NONE",
                "selected_member": str(selected_member),
                "involved_residues": involved,
                "measured_value": selected_distance,
                "lower_bound": lb,
                "upper_bound": ub,
                "violation_magnitude": violation,
                "is_violation": violation > 0,
                "passes_display_threshold": violation >= distance_threshold,
                "display_color_class": color_class(violation, distance_threshold),
                "interpretation_note": "Local inconsistency with deposited bounds; not an automatic structural error.",
                "source_saveframe": selected.get("source_saveframe"),
                "source_loop": selected.get("source_loop"),
                "atom_key_1": selected["atom_key_1"],
                "atom_key_2": selected["atom_key_2"],
            })

    if not mapped_dihedral.empty:
        for _, row in mapped_dihedral[mapped_dihedral["mapped_member"]].iterrows():
            coords = np.array([row["coord_1"], row["coord_2"], row["coord_3"], row["coord_4"]], dtype=float)
            angle = compute_dihedral(coords)
            lb = float(row["lower_bound"])
            ub = float(row["upper_bound"])
            violation = dihedral_violation(angle, lb, ub)
            involved = sorted({row["residue_key_1"], row["residue_key_2"], row["residue_key_3"], row["residue_key_4"]})

            evaluated_rows.append({
                "restraint_id": str(row["restraint_id"]),
                "restraint_type": "dihedral",
                "logical_restraint_key": f"dihedral:{row['restraint_id']}",
                "ambiguity_type": "NONE",
                "selected_member": str(row.get("torsion_name", "")),
                "involved_residues": involved,
                "measured_value": angle,
                "lower_bound": lb,
                "upper_bound": ub,
                "violation_magnitude": violation,
                "is_violation": violation > 0,
                "passes_display_threshold": violation >= dihedral_threshold,
                "display_color_class": color_class(violation, dihedral_threshold),
                "interpretation_note": "Local inconsistency with deposited bounds; not an automatic structural error.",
                "source_saveframe": row.get("source_saveframe"),
                "source_loop": row.get("source_loop"),
                "atom_key_1": row["atom_key_1"],
                "atom_key_2": row["atom_key_2"],
            })

    evaluated_restraints = pd.DataFrame(evaluated_rows)

    if evaluated_restraints.empty:
        violation_table = pd.DataFrame()
        residue_density_table = residue_table[["residue_key", "auth_asym_id", "auth_seq_id", "auth_comp_id"]].copy()
        residue_density_table["absolute_density"] = 0
        residue_density_table["normalized_density"] = 0.0
        residue_density_table["structure_mean_density"] = 0.0
        residue_density_table["local_violation_count"] = 0
        geometry_diagnostics = {
            "analysis_status": "completed_empty",
            "distance_evaluated": 0,
            "dihedral_evaluated": 0,
        }
        density_summary = {
            "residue_count": int(len(residue_table)),
            "logical_restraints_count": 0,
            "structure_mean_density": 0.0,
            "max_absolute_density": 0,
        }
        return evaluated_restraints, violation_table, residue_density_table, geometry_diagnostics, density_summary

    violation_table = evaluated_restraints.copy()
    violation_table["first_residue_number"] = violation_table["involved_residues"].map(
        lambda lst: int(str(lst[0]).split(":")[1]) if lst and str(lst[0]).split(":")[1].isdigit() else 10**9
    )
    violation_table = violation_table.sort_values(
        by=["violation_magnitude", "restraint_type", "first_residue_number"],
        ascending=[False, True, True],
    ).reset_index(drop=True)

    logical_per_residue: Dict[str, set] = {rk: set() for rk in residue_table["residue_key"].tolist()}
    local_violation_count: Dict[str, int] = {rk: 0 for rk in residue_table["residue_key"].tolist()}

    for _, row in evaluated_restraints.iterrows():
        logical_id = row["logical_restraint_key"]
        for rk in row["involved_residues"]:
            logical_per_residue.setdefault(rk, set()).add(logical_id)
            if bool(row["is_violation"]):
                local_violation_count[rk] = local_violation_count.get(rk, 0) + 1

    logical_total = int(evaluated_restraints["logical_restraint_key"].nunique())
    residue_count = int(len(residue_table))
    structure_mean_density = (logical_total / residue_count) if residue_count else 0.0

    rows = []
    for _, residue in residue_table.iterrows():
        rk = residue["residue_key"]
        abs_density = len(logical_per_residue.get(rk, set()))
        norm_density = (abs_density / structure_mean_density) if structure_mean_density > 0 else 0.0
        rows.append({
            "chain_id": residue["auth_asym_id"],
            "residue_number": residue["auth_seq_id"],
            "residue_name": residue["auth_comp_id"],
            "residue_key": rk,
            "absolute_density": int(abs_density),
            "normalized_density": float(norm_density),
            "structure_mean_density": float(structure_mean_density),
            "local_violation_count": int(local_violation_count.get(rk, 0)),
            "auth_asym_id": residue["auth_asym_id"],
            "auth_seq_id": residue["auth_seq_id"],
            "auth_comp_id": residue["auth_comp_id"],
            "ca_x": residue["ca_x"],
            "ca_y": residue["ca_y"],
            "ca_z": residue["ca_z"],
            "centroid_x": residue["centroid_x"],
            "centroid_y": residue["centroid_y"],
            "centroid_z": residue["centroid_z"],
        })

    residue_density_table = pd.DataFrame(rows)

    geometry_diagnostics = {
        "analysis_status": "completed",
        "distance_evaluated": int((evaluated_restraints["restraint_type"] == "distance").sum()),
        "dihedral_evaluated": int((evaluated_restraints["restraint_type"] == "dihedral").sum()),
    }

    density_summary = {
        "residue_count": residue_count,
        "logical_restraints_count": logical_total,
        "structure_mean_density": structure_mean_density,
        "max_absolute_density": int(residue_density_table["absolute_density"].max()) if not residue_density_table.empty else 0,
        "max_density_residue": residue_density_table.sort_values(["absolute_density", "local_violation_count"], ascending=False).iloc[0]["residue_key"] if not residue_density_table.empty else None,
    }

    return evaluated_restraints, violation_table, residue_density_table, geometry_diagnostics, density_summary


evaluated_restraints, violation_table, residue_density_table, geometry_diagnostics, density_summary = evaluate_restraints(
    mapped_distance_restraints,
    mapped_dihedral_restraints,
    residue_table,
    mapping_report,
    runtime_parameters,
)

print("Geometry diagnostics:")
display(pd.DataFrame([geometry_diagnostics]))
print("Density summary:")
display(pd.DataFrame([density_summary]))
if not violation_table.empty:
    display(violation_table.head(10))


Geometry diagnostics:


,analysis_status,distance_evaluated,dihedral_evaluated
0,completed,1150,149


Density summary:


,residue_count,logical_restraints_count,structure_mean_density,max_absolute_density,max_density_residue
0,82,1299,15.841463,81,A:12:ARG:0


,restraint_id,restraint_type,logical_restraint_key,ambiguity_type,selected_member,involved_residues,measured_value,lower_bound,upper_bound,violation_magnitude,is_violation,passes_display_threshold,display_color_class,interpretation_note,source_saveframe,source_loop,atom_key_1,atom_key_2,first_residue_number
0,144,dihedral,dihedral:144,NONE,PHI,"[A:78:ASN:0, A:79:MET:0]",-92.720788,-89.97,-49.97,2.750788,True,False,low,Local inconsistency with deposited bounds; not...,XPLOR-NIH_dihedral_angle_restraints_1,_Torsion_angle_constraint,"(A, 78, ASN, C)","(A, 79, MET, N)",78
1,63,dihedral,dihedral:63,NONE,PHI,"[A:34:SER:0, A:35:ASP:0]",-94.549124,-92.10,-52.10,2.449124,True,False,low,Local inconsistency with deposited bounds; not...,XPLOR-NIH_dihedral_angle_restraints_1,_Torsion_angle_constraint,"(A, 34, SER, C)","(A, 35, ASP, N)",34
2,105,dihedral,dihedral:105,NONE,PHI,"[A:58:VAL:0, A:59:LYS:0]",-103.486890,-102.10,-35.76,1.386890,True,False,low,Local inconsistency with deposited bounds; not...,XPLOR-NIH_dihedral_angle_restraints_1,_Torsion_angle_constraint,"(A, 58, VAL, C)","(A, 59, LYS, N)",58
3,59,dihedral,dihedral:59,NONE,PHI,"[A:32:LEU:0, A:33:TYR:0]",-85.599778,-84.34,-44.34,1.259778,True,False,low,Local inconsistency with deposited bounds; not...,XPLOR-NIH_dihedral_angle_restraints_1,_Torsion_angle_constraint,"(A, 32, LEU, C)","(A, 33, TYR, N)",32
4,88,dihedral,dihedral:88,NONE,PHI,"[A:49:LYS:0, A:50:ILE:0]",-86.458617,-85.58,-45.58,0.878617,True,False,low,Local inconsistency with deposited bounds; not...,XPLOR-NIH_dihedral_angle_restraints_1,_Torsion_angle_constraint,"(A, 49, LYS, C)","(A, 50, ILE, N)",49
5,1128,distance,distance:1128,OR,.,"[A:77:GLN:0, A:80:THR:0]",5.865710,1.80,5.03,0.835710,True,True,medium,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_1,_Gen_dist_constraint,"(A, 77, GLN, H)","(A, 80, THR, HG23)",77
6,398,distance,distance:398,OR,.,"[A:18:LEU:0, A:49:LYS:0]",5.616489,1.80,4.83,0.786489,True,True,medium,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_1,_Gen_dist_constraint,"(A, 18, LEU, HD13)","(A, 49, LYS, H)",18
7,331,distance,distance:331,OR,.,"[A:15:MET:0, A:18:LEU:0]",5.694865,1.80,4.95,0.744865,True,True,medium,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_1,_Gen_dist_constraint,"(A, 15, MET, HB3)","(A, 18, LEU, HD11)",15
8,141,dihedral,dihedral:141,NONE,PSI,"[A:77:GLN:0, A:78:ASN:0]",-21.932560,-62.62,-22.62,0.687440,True,False,low,Local inconsistency with deposited bounds; not...,XPLOR-NIH_dihedral_angle_restraints_1,_Torsion_angle_constraint,"(A, 77, GLN, N)","(A, 77, GLN, CA)",77
9,225,distance,distance:225,OR,.,"[A:11:LEU:0, A:69:VAL:0]",7.277156,1.80,6.60,0.677156,True,True,medium,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_1,_Gen_dist_constraint,"(A, 11, LEU, HD21)","(A, 69, VAL, HA)",11


## How To Interpret The Tables

- Mapping coverage is computed over logical restraints. Ambiguous `OR` groups count as one logical restraint and map when at least one candidate member is evaluable.
- Restraint density is a proxy for experimental coverage, not a confidence score.
- Violations indicate local inconsistency with deposited bounds and may reflect ambiguity, dynamics, or refinement tradeoffs. They are not automatic structure errors.
- If no thresholded violations are shown, that is not a validation verdict.

In [8]:
def value_to_color(value: float) -> str:
    # blue (low) -> yellow (mid) -> red (high)
    x = max(0.0, min(2.0, float(value))) / 2.0
    if x < 0.5:
        t = x / 0.5
        r, g, b = int(60 + t * (245 - 60)), int(120 + t * (210 - 120)), int(245 - t * (100))
    else:
        t = (x - 0.5) / 0.5
        r, g, b = int(245), int(210 - t * 170), int(145 - t * 120)
    return f"#{r:02X}{g:02X}{b:02X}"


def model_uri_and_format(model_source: Dict[str, Any]) -> Tuple[str, str]:
    # Match homodimer_diagnostic.ipynb for remote/cache mode: Mol* loads a browser-fetchable URL.
    visualization_url = model_source.get("visualization_url") or model_source.get("source_url")
    if model_source.get("mode") in {"remote", "cache"} and visualization_url:
        return str(visualization_url), "mmcif"

    # Local-file mode cannot hand browser-side Mol* an arbitrary filesystem path.
    # Use a data URI fallback only for user-supplied local files.
    path = Path(model_source["path"])
    text = path.read_text(encoding="utf-8", errors="replace")
    encoded = base64.b64encode(text.encode("utf-8")).decode("ascii")
    return f"data:text/plain;base64,{encoded}", "mmcif"


def render_mvs_state(state: Any, label: str, width: int = 960, height: int = 560) -> None:
    html = state.molstar_html()
    encoded = base64.b64encode(html.encode("utf-8")).decode("ascii")
    display(HTML(f"<div style='margin:8px 0 4px; font-weight:600;'>{label}</div>"))
    display(IFrame(src=f"data:text/html;base64,{encoded}", width=width, height=height))


def build_global_mvs_state(model_source: Dict[str, Any], residue_density: pd.DataFrame, violations: pd.DataFrame, config: Dict[str, Any]) -> Any:
    if not MVS_AVAILABLE:
        raise RuntimeError("molviewspec unavailable")

    model_uri, model_fmt = model_uri_and_format(model_source)
    builder = mvs.create_builder()
    structure = builder.download(url=model_uri).parse(format=model_fmt).model_structure()

    base = structure.component(selector="polymer").representation(type="cartoon")
    base.color(color="#B9BDC9")

    value_col = "normalized_density" if config.get("density_mode", "normalized") == "normalized" else "absolute_density"
    for _, row in residue_density.iterrows():
        seq_raw = str(row["auth_seq_id"])
        if not seq_raw.lstrip("-").isdigit():
            continue
        seq = int(seq_raw)
        comp = structure.component(
            selector=mvs.ComponentExpression(auth_asym_id=str(row["auth_asym_id"]), auth_seq_id=seq)
        )
        comp.representation(type="cartoon").color(color=value_to_color(float(row[value_col])))

    if not violations.empty and int(config.get("max_visible_restraints", 250)) > 0:
        prim = structure.primitives()
        maxn = int(config.get("max_visible_restraints", 250))
        rows = violations[(violations["restraint_type"] == "distance") & (violations["passes_display_threshold"])].head(maxn)
        for _, row in rows.iterrows():
            a1 = row["atom_key_1"]
            a2 = row["atom_key_2"]
            if not str(a1[1]).lstrip("-").isdigit() or not str(a2[1]).lstrip("-").isdigit():
                continue
            prim.distance(
                start=mvs.ComponentExpression(auth_asym_id=str(a1[0]), auth_seq_id=int(a1[1]), auth_atom_id=str(a1[3])),
                end=mvs.ComponentExpression(auth_asym_id=str(a2[0]), auth_seq_id=int(a2[1]), auth_atom_id=str(a2[3])),
                color="#D62728",
                radius=0.08,
                dash_length=0.2,
            )

    return builder.get_state()


global_mvs_state = None
if MVS_AVAILABLE:
    try:
        global_mvs_state = build_global_mvs_state(model_source, residue_density_table, violation_table, CONFIG)
        render_mvs_state(global_mvs_state, "Global density view with thresholded distance violations")
    except Exception as exc:
        log_event("warning", "global MolViewSpec render failed", error=str(exc))
        display(HTML(f"<b>Global view fallback:</b> {exc}"))
        display(residue_density_table.head(20))
else:
    display(HTML("<b>Global view fallback:</b> molviewspec is not installed."))
    display(residue_density_table.head(20))


In [9]:
def select_default_residue(residue_density: pd.DataFrame) -> Optional[str]:
    if residue_density.empty:
        return None
    ranked = residue_density.sort_values(
        ["local_violation_count", "absolute_density", "normalized_density"],
        ascending=[False, False, False],
    )
    return str(ranked.iloc[0]["residue_key"])


selected_residue_key = select_default_residue(residue_density_table)
selection_state = {
    "selected_residue_key": selected_residue_key,
    "source": "default_density_rank",
    "local_view_mode": runtime_parameters.get("local_view_mode", "distance"),
}

if WIDGETS_AVAILABLE and not residue_density_table.empty:
    options = residue_density_table["residue_key"].tolist()
    dropdown = widgets.Dropdown(options=options, value=selected_residue_key, description="Residue:")
    mode_toggle = widgets.ToggleButtons(options=["distance", "dihedral"], value=selection_state["local_view_mode"], description="Mode:")

    def _sync(change):
        selection_state["selected_residue_key"] = dropdown.value
        selection_state["local_view_mode"] = mode_toggle.value

    dropdown.observe(_sync, names="value")
    mode_toggle.observe(_sync, names="value")
    display(widgets.HBox([dropdown, mode_toggle]))

display(pd.DataFrame([selection_state]))


,selected_residue_key,source,local_view_mode
0,A:50:ILE:0,default_density_rank,distance


In [10]:
def residue_center(row: pd.Series) -> np.ndarray:
    if not np.isnan(row["ca_x"]):
        return np.array([row["ca_x"], row["ca_y"], row["ca_z"]], dtype=float)
    return np.array([row["centroid_x"], row["centroid_y"], row["centroid_z"]], dtype=float)


def compute_local_context(
    selection_state: Dict[str, Any],
    residue_density: pd.DataFrame,
    evaluated_restraints: pd.DataFrame,
    radius: float,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if residue_density.empty or not selection_state.get("selected_residue_key"):
        return pd.DataFrame(), pd.DataFrame()

    selected_key = selection_state["selected_residue_key"]
    selected_rows = residue_density[residue_density["residue_key"] == selected_key]
    if selected_rows.empty:
        return pd.DataFrame(), pd.DataFrame()

    selected = selected_rows.iloc[0]
    center = residue_center(selected)

    density = residue_density.copy()
    density["distance_to_selected"] = density.apply(lambda row: float(np.linalg.norm(residue_center(row) - center)), axis=1)
    density["in_local_radius"] = density["distance_to_selected"] <= float(radius)
    local_context = density[density["in_local_radius"]].sort_values("distance_to_selected").reset_index(drop=True)

    if evaluated_restraints.empty:
        return local_context, pd.DataFrame()

    local_keys = set(local_context["residue_key"].tolist())
    filtered = evaluated_restraints[
        evaluated_restraints["involved_residues"].map(lambda residues: bool(set(residues) & local_keys))
    ].copy()

    return local_context, filtered


local_context_table, local_restraints_table = compute_local_context(
    selection_state,
    residue_density_table,
    evaluated_restraints,
    runtime_parameters["local_context_radius"],
)

display(local_context_table.head(15))
if not local_restraints_table.empty:
    display(local_restraints_table.head(20))


,chain_id,residue_number,residue_name,residue_key,absolute_density,normalized_density,structure_mean_density,local_violation_count,auth_asym_id,auth_seq_id,auth_comp_id,ca_x,ca_y,ca_z,centroid_x,centroid_y,centroid_z,distance_to_selected,in_local_radius
0,A,50,ILE,A:50:ILE:0,75,4.734411,15.841463,23,A,50,ILE,2.125,7.525,1.984,2.030158,6.352105,1.977842,0.000000,True
1,A,51,LYS,A:51:LYS:0,50,3.156274,15.841463,7,A,51,LYS,3.767,9.733,4.618,6.373864,9.869727,5.339273,3.809119,True
2,A,49,LYS,A:49:LYS:0,34,2.146266,15.841463,8,A,49,LYS,3.426,9.656,-0.902,3.735864,10.802045,-2.931727,3.816118,True
3,A,47,VAL,A:47:VAL:0,46,2.903772,15.841463,6,A,47,VAL,6.624,5.747,1.319,7.509063,5.398000,1.989625,4.883084,True
4,A,53,LEU,A:53:LEU:0,38,2.398768,15.841463,11,A,53,LEU,-1.012,11.353,2.311,-2.251053,10.322316,1.492947,4.959968,True
5,A,52,GLU,A:52:GLU:0,24,1.515012,15.841463,5,A,52,GLU,2.463,12.820,2.847,2.469000,13.703333,1.623133,5.375504,True
6,A,48,VAL,A:48:VAL:0,30,1.893764,15.841463,4,A,48,VAL,6.958,9.434,0.525,7.706688,10.131937,-0.028625,5.397300,True
7,A,46,LEU,A:46:LEU:0,50,3.156274,15.841463,12,A,46,LEU,5.138,4.994,-2.125,4.417474,3.577789,-2.911579,5.689289,True


,restraint_id,restraint_type,logical_restraint_key,ambiguity_type,selected_member,involved_residues,measured_value,lower_bound,upper_bound,violation_magnitude,is_violation,passes_display_threshold,display_color_class,interpretation_note,source_saveframe,source_loop,atom_key_1,atom_key_2
42,43,distance,distance:43,OR,.,"[A:42:ASP:0, A:46:LEU:0]",3.174683,2.8,3.30,0.000000,False,False,none,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_2,_Gen_dist_constraint,"(A, 42, ASP, O)","(A, 46, LEU, N)"
43,44,distance,distance:44,OR,.,"[A:42:ASP:0, A:46:LEU:0]",2.231743,1.8,2.20,0.031743,True,False,low,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_2,_Gen_dist_constraint,"(A, 42, ASP, O)","(A, 46, LEU, H)"
45,46,distance,distance:46,OR,.,"[A:43:ALA:0, A:47:VAL:0]",2.856889,2.8,3.30,0.000000,False,False,none,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_2,_Gen_dist_constraint,"(A, 43, ALA, O)","(A, 47, VAL, N)"
57,58,distance,distance:58,OR,.,"[A:49:LYS:0, A:53:LEU:0]",2.274945,1.8,2.20,0.074945,True,False,low,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_2,_Gen_dist_constraint,"(A, 49, LYS, O)","(A, 53, LEU, H)"
59,60,distance,distance:60,OR,.,"[A:50:ILE:0, A:54:THR:0]",2.135469,1.8,2.20,0.000000,False,False,none,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_2,_Gen_dist_constraint,"(A, 50, ILE, O)","(A, 54, THR, H)"
60,61,distance,distance:61,OR,.,"[A:51:LYS:0, A:55:GLY:0]",1.978954,1.8,2.20,0.000000,False,False,none,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_2,_Gen_dist_constraint,"(A, 51, LYS, O)","(A, 55, GLY, H)"
98,99,distance,distance:99,OR,.,"[A:53:LEU:0, A:7:VAL:0]",5.001048,1.8,4.83,0.171048,True,False,low,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_1,_Gen_dist_constraint,"(A, 7, VAL, HA)","(A, 53, LEU, HD21)"
103,104,distance,distance:104,OR,.,"[A:53:LEU:0, A:7:VAL:0]",4.375946,1.8,5.37,0.000000,False,False,none,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_1,_Gen_dist_constraint,"(A, 7, VAL, HG13)","(A, 53, LEU, HD22)"
196,197,distance,distance:197,OR,.,"[A:10:LYS:0, A:53:LEU:0]",6.105284,1.8,5.95,0.155284,True,False,low,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_1,_Gen_dist_constraint,"(A, 10, LYS, HB3)","(A, 53, LEU, HB2)"
197,198,distance,distance:198,NONE,.,"[A:10:LYS:0, A:53:LEU:0]",6.139266,1.8,5.95,0.189266,True,False,low,Local inconsistency with deposited bounds; not...,XPLOR-NIH_distance_restraints_1,_Gen_dist_constraint,"(A, 10, LYS, HG3)","(A, 53, LEU, HG)"


In [11]:
def build_local_mvs_state(
    model_source: Dict[str, Any],
    local_context: pd.DataFrame,
    local_restraints: pd.DataFrame,
    selection_state: Dict[str, Any],
    params: Dict[str, Any],
) -> Any:
    if not MVS_AVAILABLE:
        raise RuntimeError("molviewspec unavailable")

    model_uri, model_fmt = model_uri_and_format(model_source)
    builder = mvs.create_builder()
    structure = builder.download(url=model_uri).parse(format=model_fmt).model_structure()

    structure.component(selector="polymer").representation(type="cartoon").color(color="#D9D9E3")

    selected_key = selection_state.get("selected_residue_key")
    view_mode = selection_state.get("local_view_mode", "distance")

    for _, row in local_context.iterrows():
        seq_raw = str(row["auth_seq_id"])
        if not seq_raw.lstrip("-").isdigit():
            continue
        seq = int(seq_raw)
        is_selected = row["residue_key"] == selected_key
        color = "#6C8EF5" if not is_selected else "#F2A900"
        rep = structure.component(
            selector=mvs.ComponentExpression(auth_asym_id=str(row["auth_asym_id"]), auth_seq_id=seq)
        ).representation(type="cartoon")
        rep.color(color=color)

    prim = structure.primitives()
    if not local_restraints.empty:
        table = local_restraints.copy()
        if view_mode == "distance":
            table = table[table["restraint_type"] == "distance"]
            table = table.sort_values("violation_magnitude", ascending=False)
            table = table.head(int(params.get("max_visible_restraints", 250)))
            for _, row in table.iterrows():
                a1 = row["atom_key_1"]
                a2 = row["atom_key_2"]
                if not str(a1[1]).lstrip("-").isdigit() or not str(a2[1]).lstrip("-").isdigit():
                    continue
                color = "#D62728" if row["passes_display_threshold"] else "#9AA0A6"
                prim.distance(
                    start=mvs.ComponentExpression(auth_asym_id=str(a1[0]), auth_seq_id=int(a1[1]), auth_atom_id=str(a1[3])),
                    end=mvs.ComponentExpression(auth_asym_id=str(a2[0]), auth_seq_id=int(a2[1]), auth_atom_id=str(a2[3])),
                    color=color,
                    radius=0.10,
                    dash_length=0.2,
                )
        else:
            # Dihedral mode: emphasize residues participating in dihedral restraints.
            dih = table[table["restraint_type"] == "dihedral"]
            touched = set()
            for _, row in dih.iterrows():
                touched.update(row["involved_residues"])
            for rk in touched:
                parts = rk.split(":")
                if len(parts) < 4:
                    continue
                chain, seq_raw = parts[0], parts[1]
                if not seq_raw.lstrip("-").isdigit():
                    continue
                seq = int(seq_raw)
                structure.component(
                    selector=mvs.ComponentExpression(auth_asym_id=chain, auth_seq_id=seq)
                ).representation(type="ball_and_stick").color(color="#B455E0")

    return builder.get_state()


local_mvs_state = None
if MVS_AVAILABLE and not local_context_table.empty:
    try:
        local_mvs_state = build_local_mvs_state(
            model_source,
            local_context_table,
            local_restraints_table,
            selection_state,
            runtime_parameters,
        )
        render_mvs_state(local_mvs_state, f"Local evidence view ({selection_state.get('local_view_mode', 'distance')})")
    except Exception as exc:
        log_event("warning", "local MolViewSpec render failed", error=str(exc))
        display(HTML(f"<b>Local view fallback:</b> {exc}"))
        display(local_context_table)
        display(local_restraints_table.head(20))
else:
    display(HTML("<b>Local view fallback:</b> local context unavailable or molviewspec missing."))


In [12]:
OUTPUT_DIR = PROJECT_ROOT / "specs/nmr_restraints/outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def export_outputs(
    output_dir: Path,
    violation_table: pd.DataFrame,
    residue_density_table: pd.DataFrame,
    mapping_report: Dict[str, Any],
    evaluated_restraints: pd.DataFrame,
    provenance: pd.DataFrame,
    runtime_parameters: Dict[str, Any],
) -> pd.DataFrame:
    exports = []

    run_label = datetime.now().strftime("%Y%m%d_%H%M%S")

    if not violation_table.empty:
        vt = violation_table.copy()
        vt["pdb_id_or_label"] = input_request.get("pdb_id") or "local"
        vt["model_index"] = runtime_parameters["model_index"]
        vt["mapping_coverage"] = mapping_report.get("logical_mapping_coverage", np.nan)
        vt["ruo_note"] = "Research-use only exploratory evidence; not a validation verdict."
        path = output_dir / f"violation_table_{run_label}.csv"
        vt.to_csv(path, index=False)
        exports.append({"artifact": "violation_table_csv", "path": str(path), "rows": len(vt)})

    if not residue_density_table.empty:
        path = output_dir / f"residue_density_{run_label}.csv"
        residue_density_table.to_csv(path, index=False)
        exports.append({"artifact": "residue_density_csv", "path": str(path), "rows": len(residue_density_table)})

    if not evaluated_restraints.empty:
        path = output_dir / f"evaluated_restraints_{run_label}.csv"
        evaluated_restraints.to_csv(path, index=False)
        exports.append({"artifact": "evaluated_restraints_csv", "path": str(path), "rows": len(evaluated_restraints)})

    map_path = output_dir / f"mapping_report_{run_label}.json"
    map_path.write_text(json.dumps(mapping_report, indent=2), encoding="utf-8")
    exports.append({"artifact": "mapping_report_json", "path": str(map_path), "rows": 1})

    prov_path = output_dir / f"provenance_{run_label}.csv"
    provenance.to_csv(prov_path, index=False)
    exports.append({"artifact": "provenance_csv", "path": str(prov_path), "rows": len(provenance)})

    return pd.DataFrame(exports)


export_manifest = export_outputs(
    OUTPUT_DIR,
    violation_table,
    residue_density_table,
    mapping_report,
    evaluated_restraints,
    provenance,
    runtime_parameters,
)

display(export_manifest)


,artifact,path,rows
0,violation_table_csv,/Users/mitsenkov/PycharmProjects/InsightFold/s...,1299
1,residue_density_csv,/Users/mitsenkov/PycharmProjects/InsightFold/s...,82
2,evaluated_restraints_csv,/Users/mitsenkov/PycharmProjects/InsightFold/s...,1299
3,mapping_report_json,/Users/mitsenkov/PycharmProjects/InsightFold/s...,1
4,provenance_csv,/Users/mitsenkov/PycharmProjects/InsightFold/s...,2


## Validation Snapshot (Lightweight Construction Check)

This section captures implementation-stage evidence only. It is **not** the full execution-validation lifecycle gate.

Key caveats remain: RUO scope, mapping-threshold interpretation, and parser/visualization behavior should still be rechecked in the dedicated execution-validation stage.

In [13]:
runtime_seconds = time.time() - RUN_START

selected_local_residue = selection_state.get("selected_residue_key")

top_violations_snapshot = []
if not violation_table.empty:
    cols = [
        "restraint_type",
        "restraint_id",
        "violation_magnitude",
        "passes_display_threshold",
        "selected_member",
        "involved_residues",
    ]
    top_violations_snapshot = violation_table[cols].head(5).to_dict(orient="records")

validation_snapshot = {
    "fixture": "happy-path-9l1v",
    "parser": restraint_diagnostics.get("parser"),
    "mapping_coverage_logical": mapping_report.get("logical_mapping_coverage"),
    "mapping_status": mapping_report.get("threshold_status"),
    "distance_rows": restraint_diagnostics.get("distance_rows"),
    "distance_logical": restraint_diagnostics.get("distance_logical"),
    "distance_or_logical": restraint_diagnostics.get("distance_or_logical"),
    "torsion_rows": restraint_diagnostics.get("torsion_rows"),
    "torsion_logical": restraint_diagnostics.get("torsion_logical"),
    "distance_missing_bounds": restraint_diagnostics.get("distance_missing_bounds"),
    "density_summary": density_summary,
    "selected_local_residue": selected_local_residue,
    "top_violations": top_violations_snapshot,
    "runtime_seconds": runtime_seconds,
    "mvs_global_state_present": global_mvs_state is not None,
    "mvs_local_state_present": local_mvs_state is not None,
    "ruo_scope": "Research-use exploratory evidence viewer only.",
}

snapshot_path = PROJECT_ROOT / "specs/nmr_restraints/fixtures/synthetic/happy_path_snapshot_latest.json"
snapshot_path.parent.mkdir(parents=True, exist_ok=True)
snapshot_path.write_text(json.dumps(validation_snapshot, indent=2), encoding="utf-8")

print("Validation snapshot file:", snapshot_path)
display(pd.DataFrame([{
    "mapping_coverage": validation_snapshot["mapping_coverage_logical"],
    "mapping_status": validation_snapshot["mapping_status"],
    "selected_local_residue": validation_snapshot["selected_local_residue"],
    "runtime_seconds": round(validation_snapshot["runtime_seconds"], 3),
    "mvs_global_state_present": validation_snapshot["mvs_global_state_present"],
    "mvs_local_state_present": validation_snapshot["mvs_local_state_present"],
}]))

print("Top violations (first five):")
display(pd.DataFrame(top_violations_snapshot))

print("Notebook log events:")
if notebook_log:
    display(pd.DataFrame(notebook_log))
else:
    print("No warnings/errors logged.")


Validation snapshot file: /Users/mitsenkov/PycharmProjects/InsightFold/specs/nmr_restraints/fixtures/synthetic/happy_path_snapshot_latest.json


,mapping_coverage,mapping_status,selected_local_residue,runtime_seconds,mvs_global_state_present,mvs_local_state_present
0,1.0,normal,A:50:ILE:0,0.841,True,True


Top violations (first five):


,restraint_type,restraint_id,violation_magnitude,passes_display_threshold,selected_member,involved_residues
0,dihedral,144,2.750788,False,PHI,"[A:78:ASN:0, A:79:MET:0]"
1,dihedral,63,2.449124,False,PHI,"[A:34:SER:0, A:35:ASP:0]"
2,dihedral,105,1.386890,False,PHI,"[A:58:VAL:0, A:59:LYS:0]"
3,dihedral,59,1.259778,False,PHI,"[A:32:LEU:0, A:33:TYR:0]"
4,dihedral,88,0.878617,False,PHI,"[A:49:LYS:0, A:50:ILE:0]"


Notebook log events:
No warnings/errors logged.
